# Korean Chatbot - Stage 1
1. 한국어 위키피디아로 토크나이저 + 사전학습
2. KoAlpaca로 파인튜닝

## 0. 환경 설정

In [ ]:
!pip install datasets tqdm -q

In [ ]:
import os, sys, shutil, importlib

REPO_URL  = "https://github.com/kkkk2058/korean-chatbot"
REPO_DIR  = "korean-chatbot"
STAGE_DIR = f"{REPO_DIR}/stage1_from_scratch"
DRIVE_DIR = "/content/drive/MyDrive/korean_chatbot"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
else:
    !git -C $REPO_DIR pull

sys.path.insert(0, STAGE_DIR)
print("레포 준비 완료")

## 1. 토크나이저 (한국어 위키피디아)

In [ ]:
import src.tokenizer
importlib.reload(src.tokenizer)
from src.tokenizer import BPETokenizer
import config

TOKENIZER_PATH = "tokenizer.json"

# Drive에 저장된 토크나이저 있으면 복사
if not os.path.exists(TOKENIZER_PATH) and os.path.exists(f"{DRIVE_DIR}/tokenizer.json"):
    shutil.copy(f"{DRIVE_DIR}/tokenizer.json", TOKENIZER_PATH)
    print("Drive에서 tokenizer.json 복사 완료")

tokenizer = BPETokenizer()

if os.path.exists(TOKENIZER_PATH):
    tokenizer.load(TOKENIZER_PATH)
    print(f"토크나이저 로드 완료. vocab size = {len(tokenizer.vocab)}")
else:
    from datasets import load_dataset
    from tqdm.notebook import tqdm

    print("한국어 위키피디아 로드 중... (시간이 걸릴 수 있어요)")
    wiki = load_dataset("wikimedia/wikipedia", "20231101.ko", split="train")

    # 문단 단위로 분리해서 코퍼스 구성
    corpus = []
    for row in tqdm(wiki, desc="위키 코퍼스 변환 중"):
        for line in row["text"].split("\n"):
            line = line.strip()
            if len(line) > 20:   # 너무 짧은 줄 제외
                corpus.append(line)
    print(f"코퍼스 문장 수: {len(corpus):,}")

    tokenizer.train(corpus, vocab_size=config.VOCAB_SIZE)
    tokenizer.save(TOKENIZER_PATH)
    print(f"토크나이저 저장 완료. vocab size = {len(tokenizer.vocab)}")

## 2. 모델 초기화

In [ ]:
import torch
from src.model import Transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = Transformer(
    vocab_size=len(tokenizer.vocab),
    d_model=config.D_MODEL,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    max_seq_len=config.MAX_SEQ_LEN,
    dropout=config.DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"파라미터 수: {total_params:,} ({total_params/1e6:.1f}M)")

## 3. 사전학습 (한국어 위키피디아)
언어 자체를 먼저 학습. 다음 토큰 예측.

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.notebook import tqdm
import time

PAD_ID = tokenizer.vocab["<pad>"]
BOS_ID = tokenizer.vocab["<s>"]
EOS_ID = tokenizer.vocab["</s>"]


class TextDataset(Dataset):
    """일반 텍스트 → 다음 토큰 예측용 데이터셋"""
    def __init__(self, texts, tokenizer, max_seq_len):
        self.samples = []
        for text in tqdm(texts, desc="데이터셋 변환 중"):
            ids = [BOS_ID] + tokenizer.encode(text) + [EOS_ID]
            if len(ids) > 1:
                self.samples.append(ids[: max_seq_len + 1])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        return torch.tensor(ids[:-1], dtype=torch.long), torch.tensor(ids[1:], dtype=torch.long)


def collate_fn(batch):
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    pad = lambda t: torch.nn.functional.pad(t, (0, max_len - t.size(0)), value=PAD_ID)
    return torch.stack([pad(x) for x in xs]), torch.stack([pad(y) for y in ys])


# 위키피디아 로드 (토크나이저 셀에서 이미 로드했으면 재사용)
if 'wiki' not in dir():
    from datasets import load_dataset
    wiki = load_dataset("wikimedia/wikipedia", "20231101.ko", split="train")

wiki_texts = []
for row in tqdm(wiki, desc="위키 텍스트 수집"):
    for line in row["text"].split("\n"):
        line = line.strip()
        if len(line) > 20:
            wiki_texts.append(line)

t0 = time.time()
pretrain_dataset = TextDataset(wiki_texts, tokenizer, max_seq_len=config.MAX_SEQ_LEN)
train_size = int(len(pretrain_dataset) * 0.95)
val_size   = len(pretrain_dataset) - train_size
pretrain_train, pretrain_val = random_split(pretrain_dataset, [train_size, val_size])

pretrain_loader     = DataLoader(pretrain_train, batch_size=config.PRETRAIN_BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
pretrain_val_loader = DataLoader(pretrain_val,   batch_size=config.PRETRAIN_BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
print(f"사전학습 데이터: train {len(pretrain_train):,} / val {len(pretrain_val):,} / 배치 {len(pretrain_loader):,}  ({time.time()-t0:.1f}초)")

In [ ]:
import math, time
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast, GradScaler
from IPython.display import display, clear_output

PRETRAIN_CKPT = "pretrain_checkpoint.pt"

criterion  = torch.nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer  = torch.optim.AdamW(model.parameters(), lr=config.PRETRAIN_LR)
scaler     = GradScaler()   # FP16 mixed precision → VRAM 절반
total_steps = (len(pretrain_loader) // config.PRETRAIN_GRAD_ACCUM) * config.PRETRAIN_EPOCHS
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

# Drive 체크포인트 복사
if not os.path.exists(PRETRAIN_CKPT) and os.path.exists(f"{DRIVE_DIR}/pretrain_checkpoint.pt"):
    shutil.copy(f"{DRIVE_DIR}/pretrain_checkpoint.pt", PRETRAIN_CKPT)
    print("Drive에서 pretrain_checkpoint.pt 복사 완료")

start_epoch = 0
if os.path.exists(PRETRAIN_CKPT):
    ckpt = torch.load(PRETRAIN_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = ckpt["epoch"] + 1
    print(f"체크포인트 로드. epoch {start_epoch}부터 재개")

history = {"epoch": [], "train": [], "val": []}
train_start = time.time()
fig, ax = plt.subplots(figsize=(8, 4))

def update_plot(title):
    ax.cla()
    ax.plot(history["epoch"], history["train"], "b-o", markersize=4, label="train")
    ax.plot(history["epoch"], history["val"],   "r-o", markersize=4, label="val")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True)
    elapsed = time.time() - train_start
    done = len(history["epoch"])
    eta_str = f"  |  ETA {(elapsed/done)*(config.PRETRAIN_EPOCHS - start_epoch - done)/60:.1f}분" if done > 0 else ""
    fig.suptitle(f"경과 {elapsed/60:.1f}분{eta_str}")
    fig.tight_layout(); clear_output(wait=True); display(fig)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, steps = 0, 0
    pbar = tqdm(loader, desc="train" if train else "val ", leave=False)
    optimizer.zero_grad()

    with torch.set_grad_enabled(train):
        for step, (x, y) in enumerate(pbar):
            x, y = x.to(device), y.to(device)
            with autocast():   # FP16
                logits = model(x)
                loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
                if train:
                    loss = loss / config.PRETRAIN_GRAD_ACCUM

            if train:
                scaler.scale(loss).backward()
                if (step + 1) % config.PRETRAIN_GRAD_ACCUM == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()

            total_loss += loss.item() * (config.PRETRAIN_GRAD_ACCUM if train else 1)
            steps += 1
            pbar.set_postfix({"loss": f"{total_loss/steps:.4f}"})

    return total_loss / steps


print("사전학습 시작!")
for epoch in range(start_epoch, config.PRETRAIN_EPOCHS):
    train_loss = run_epoch(pretrain_loader, train=True)
    val_loss   = run_epoch(pretrain_val_loader, train=False)
    elapsed    = time.time() - train_start

    history["epoch"].append(epoch + 1)
    history["train"].append(train_loss)
    history["val"].append(val_loss)
    update_plot("사전학습 Loss")
    print(f"Epoch {epoch+1:02d} | train {train_loss:.4f} | val {val_loss:.4f} | 경과 {elapsed/60:.1f}분")
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict()}, PRETRAIN_CKPT)

print("사전학습 완료!")

## 4. 파인튜닝 (KoAlpaca)
사전학습된 모델에 질문-답변 형식 학습.

In [ ]:
from datasets import load_dataset

print("KoAlpaca 로드 중...")
koalpaca = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
print(f"샘플 수: {len(koalpaca)}")
print("예시:", koalpaca[0])

In [ ]:
class InstructDataset(Dataset):
    """### 질문 / ### 답변 형식으로 파인튜닝"""
    def __init__(self, hf_dataset, tokenizer, max_seq_len):
        self.samples = []
        for row in tqdm(hf_dataset, desc="파인튜닝 데이터 변환 중"):
            inp  = row.get("input", "").strip()
            q    = f"{row['instruction'].strip()}{chr(10)+inp if inp else ''}"
            text = f"### 질문: {q}\n### 답변: {row['output'].strip()}"
            ids  = [BOS_ID] + tokenizer.encode(text) + [EOS_ID]
            if len(ids) > 1:
                self.samples.append(ids[: max_seq_len + 1])

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        ids = self.samples[idx]
        return torch.tensor(ids[:-1], dtype=torch.long), torch.tensor(ids[1:], dtype=torch.long)


t0 = time.time()
ft_dataset = InstructDataset(koalpaca, tokenizer, max_seq_len=config.MAX_SEQ_LEN)
ft_train_size = int(len(ft_dataset) * 0.9)
ft_val_size   = len(ft_dataset) - ft_train_size
ft_train, ft_val = random_split(ft_dataset, [ft_train_size, ft_val_size])

ft_loader     = DataLoader(ft_train, batch_size=config.FINETUNE_BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
ft_val_loader = DataLoader(ft_val,   batch_size=config.FINETUNE_BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
print(f"파인튜닝 데이터: train {len(ft_train):,} / val {len(ft_val):,} / 배치 {len(ft_loader):,}  ({time.time()-t0:.1f}초)")

In [ ]:
FINETUNE_CKPT = "finetune_checkpoint.pt"

# 파인튜닝은 LR 낮게, 사전학습 파라미터 유지하면서 조금만 업데이트
ft_optimizer  = torch.optim.AdamW(model.parameters(), lr=config.FINETUNE_LR)
ft_scaler     = GradScaler()
ft_total_steps = (len(ft_loader) // config.FINETUNE_GRAD_ACCUM) * config.FINETUNE_EPOCHS
ft_scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(ft_optimizer, T_max=ft_total_steps)

if not os.path.exists(FINETUNE_CKPT) and os.path.exists(f"{DRIVE_DIR}/finetune_checkpoint.pt"):
    shutil.copy(f"{DRIVE_DIR}/finetune_checkpoint.pt", FINETUNE_CKPT)
    print("Drive에서 finetune_checkpoint.pt 복사 완료")

ft_start_epoch = 0
if os.path.exists(FINETUNE_CKPT):
    ckpt = torch.load(FINETUNE_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    ft_optimizer.load_state_dict(ckpt["optimizer"])
    ft_start_epoch = ckpt["epoch"] + 1
    print(f"파인튜닝 체크포인트 로드. epoch {ft_start_epoch}부터 재개")

ft_history = {"epoch": [], "train": [], "val": []}
ft_start   = time.time()
fig2, ax2  = plt.subplots(figsize=(8, 4))

def update_ft_plot():
    ax2.cla()
    ax2.plot(ft_history["epoch"], ft_history["train"], "b-o", markersize=4, label="train")
    ax2.plot(ft_history["epoch"], ft_history["val"],   "r-o", markersize=4, label="val")
    ax2.set_title("파인튜닝 Loss"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
    ax2.legend(); ax2.grid(True)
    elapsed = time.time() - ft_start
    done = len(ft_history["epoch"])
    eta_str = f"  |  ETA {(elapsed/done)*(config.FINETUNE_EPOCHS - ft_start_epoch - done)/60:.1f}분" if done > 0 else ""
    fig2.suptitle(f"경과 {elapsed/60:.1f}분{eta_str}")
    fig2.tight_layout(); clear_output(wait=True); display(fig2)


def run_ft_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, steps = 0, 0
    pbar = tqdm(loader, desc="train" if train else "val ", leave=False)
    ft_optimizer.zero_grad()

    with torch.set_grad_enabled(train):
        for step, (x, y) in enumerate(pbar):
            x, y = x.to(device), y.to(device)
            with autocast():
                logits = model(x)
                loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
                if train:
                    loss = loss / config.FINETUNE_GRAD_ACCUM

            if train:
                ft_scaler.scale(loss).backward()
                if (step + 1) % config.FINETUNE_GRAD_ACCUM == 0:
                    ft_scaler.unscale_(ft_optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    ft_scaler.step(ft_optimizer)
                    ft_scaler.update()
                    ft_scheduler.step()
                    ft_optimizer.zero_grad()

            total_loss += loss.item() * (config.FINETUNE_GRAD_ACCUM if train else 1)
            steps += 1
            pbar.set_postfix({"loss": f"{total_loss/steps:.4f}"})

    return total_loss / steps


print("파인튜닝 시작!")
for epoch in range(ft_start_epoch, config.FINETUNE_EPOCHS):
    train_loss = run_ft_epoch(ft_loader, train=True)
    val_loss   = run_ft_epoch(ft_val_loader, train=False)
    elapsed    = time.time() - ft_start

    ft_history["epoch"].append(epoch + 1)
    ft_history["train"].append(train_loss)
    ft_history["val"].append(val_loss)
    update_ft_plot()
    print(f"Epoch {epoch+1:02d} | train {train_loss:.4f} | val {val_loss:.4f} | 경과 {elapsed/60:.1f}분")
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": ft_optimizer.state_dict()}, FINETUNE_CKPT)

print("파인튜닝 완료!")

## 5. 생성 테스트

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=200, temperature=0.8, top_k=50):
    model.eval()
    text  = f"### 질문: {prompt}\n### 답변:"
    ids   = [BOS_ID] + tokenizer.encode(text)
    x     = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        if x.size(1) >= config.MAX_SEQ_LEN:
            break
        with autocast():
            logits = model(x)[:, -1, :] / temperature
        topk_vals, _ = torch.topk(logits, top_k)
        logits[logits < topk_vals[:, -1:]] = float("-inf")
        next_id = torch.multinomial(torch.softmax(logits, dim=-1), num_samples=1)
        if next_id.item() == EOS_ID:
            break
        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist()[len(ids):])


prompts = ["한국의 수도는 어디인가요?", "파이썬이란 무엇인가요?"]
for p in prompts:
    print(f"Q: {p}")
    print(f"A: {generate(p)}")
    print()

## 6. Google Drive 저장

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(DRIVE_DIR, exist_ok=True)
for fname in ["tokenizer.json", "pretrain_checkpoint.pt", "finetune_checkpoint.pt"]:
    if os.path.exists(fname):
        shutil.copy(fname, f"{DRIVE_DIR}/{fname}")
        print(f"저장 완료: {DRIVE_DIR}/{fname}")